# CropMedic Zimbabwe v1 — GPU training
This notebook clones the `production-v1` branch, prepares the Zimbabwe PlantVillage profile, trains MobileNetV2, evaluates the official held-out PlantVillage test split, and saves the model artifacts to Google Drive.

In [ ]:
!nvidia-smi
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU detected. In Colab choose Runtime > Change runtime type > GPU.'

In [ ]:
!rm -rf /content/CropMedic
!git clone --branch production-v1 --single-branch https://github.com/macdonaldchigutiro/CropMedic.git /content/CropMedic
%cd /content/CropMedic/crop_medic_backend/training

In [ ]:
!python -m pip install -q -r requirements-training.txt

In [ ]:
!python prepare_plantvillage.py --profile zimbabwe-v1 --output-dir data

In [ ]:
!python train.py --data-dir data --output-dir artifacts --batch-size 32 --head-epochs 12 --fine-epochs 12

In [ ]:
import json
from pathlib import Path
metrics = json.loads(Path('artifacts/metrics.json').read_text())
print('Held-out test accuracy:', metrics['test_accuracy'])
weak = sorted(metrics['per_class'].items(), key=lambda item: item[1]['f1'])
print('\nLowest-F1 classes:')
for name, values in weak[:10]:
    print(f"{name}: precision={values['precision']:.4f}, recall={values['recall']:.4f}, f1={values['f1']:.4f}, support={values['support']}")

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
destination = Path('/content/drive/MyDrive/CropMedic/zimbabwe-v1-artifacts')
destination.mkdir(parents=True, exist_ok=True)
for source in Path('artifacts').iterdir():
    target = destination / source.name
    if source.is_file():
        shutil.copy2(source, target)
print('Saved artifacts to:', destination)